# Multiple seasonalities

> In this example we will show how to perform electricity load forecasting considering a model capable of handling multiple seasonalities (MSTL).

<a href="https://colab.research.google.com/github/Nixtla/statsforecast/blob/main/nbs/examples/MultipleSeasonalities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

Some time series are generated from very low frequency data. These data generally exhibit multiple seasonalities. For example, hourly data may exhibit repeated patterns every hour (every 24 observations) or every day (every 24 * 7, hours per day, observations). This is the case for electricity load. Electricity load may vary hourly, e.g., during the evenings electricity consumption may be expected to increase. But also, the electricity load varies by week. Perhaps on weekends there is an increase in electrical activity.


In this example we will show how to model the two seasonalities of the time series to generate accurate forecasts in a short time. We will use hourly PJM electricity load data. The original data can be found [here](https://github.com/jnagura/Energy-consumption-prediction-analysis). 

## Libraries

In this example we will use the following libraries:
- [`StatsForecast`](https://github.com/Nixtla/statsforecast). Lightning ⚡️ fast forecasting with statistical and econometric models. Includes the MSTL model for multiple seasonalities.
- [`DatasetsForecast`](https://github.com/Nixtla/datasetsforecast). Used to evaluate the performance of the forecasts.
- [`Prophet`](https://github.com/facebook/prophet). Benchmark model developed by Facebook.
- [`NeuralProphet`](https://github.com/ourownstory/neural_prophet). Deep Learning version of `Prophet`. Used as benchark.

In [ ]:
%%capture
!pip install statsforecast
!pip install datasetsforecast
!pip install prophet
!pip install "neuralprophet[live]"

## Forecast using Multiple Seasonalities

### Electricity Load Data

According to the [dataset's page](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption),

> PJM Interconnection LLC (PJM) is a regional transmission organization (RTO) in the United States. It is part of the Eastern Interconnection grid operating an electric transmission system serving all or parts of Delaware, Illinois, Indiana, Kentucky, Maryland, Michigan, New Jersey, North Carolina, Ohio, Pennsylvania, Tennessee, Virginia, West Virginia, and the District of Columbia. The hourly power consumption data comes from PJM's website and are in megawatts (MW).

Let's take a look to the data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsforecast import StatsForecast

pd.plotting.register_matplotlib_converters()
plt.rc("figure", figsize=(10, 8))
plt.rc("font", size=10)

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/jnagura/Energy-consumption-prediction-analysis/master/PJM_Load_hourly.csv')
df.columns = ['ds', 'y']
df.insert(0, 'unique_id', 'PJM_Load_hourly')
df['ds'] = pd.to_datetime(df['ds'])
df = df.sort_values(['unique_id', 'ds']).reset_index(drop=True)
df.tail()

In [ ]:
StatsForecast.plot(df)

We clearly observe that the time series exhibits seasonal patterns. Moreover, the time series contains `32,896` observations, so it is necessary to use very computationally efficient methods to display them in production.

### MSTL model

The `MSTL` (Multiple Seasonal-Trend decomposition using LOESS) model, originally developed by [Kasun Bandara, Rob J Hyndman and Christoph Bergmeir](https://arxiv.org/abs/2107.13462), decomposes the time series in multiple seasonalities using a Local Polynomial Regression (LOESS). Then it forecasts the trend using a custom non-seasonal model and each seasonality using a `SeasonalNaive` model.

`StatsForecast` contains a fast implementation of the `MSTL` model. Also, the decomposition of the time series can be calculated.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import MSTL, AutoARIMA, SeasonalNaive
from statsforecast.utils import AirPassengers as ap

First we must define the model parameters. As mentioned before, the electricity load presents seasonalities every 24 hours (Hourly) and every 24 * 7 (Daily) hours. Therefore, we will use `[24, 24 * 7]` as the seasonalities that the MSTL model receives. We must also specify the manner in which the trend will be forecasted. In this case we will use the `AutoARIMA` model.

In [ ]:
mstl = MSTL(
    season_length=[24, 24 * 7], # seasonalities of the time series 
    trend_forecaster=AutoARIMA() # model used to forecast trend
)

Once the model is instantiated, we have to instantiate the `StatsForecast` class to create forecasts. 

In [ ]:
sf = StatsForecast(
    models=[mstl], # model used to fit each time series 
    freq='H', # frequency of the data
)

#### Fit the model

Afer that, we just have to use the `fit` method to fit each model to each time series.

In [ ]:
sf = sf.fit(df=df)

#### Decompose the time series in multiple seasonalities

Once the model is fitted, we can access the decomposition using the `fitted_` attribute of `StatsForecast`. This attribute stores all relevant information of the fitted models for each of the time series. 


In this case we are fitting a single model for a single time series, so by accessing the fitted_ location [0, 0] we will find the relevant information of our model. The `MSTL` class generates a `model_` attribute that contains the way the series was decomposed.

In [ ]:
sf.fitted_[0, 0].model_

Let's look graphically at the different components of the time series.

In [ ]:
sf.fitted_[0, 0].model_.tail(24 * 28).plot(subplots=True, grid=True)
plt.tight_layout()
plt.show()

We observe that there is a clear trend towards the high (orange line). This component would be predicted with the `AutoARIMA` model. We can also observe that every 24 hours and every `24 * 7` hours there is a very well defined pattern. These two components will be forecast separately using a `SeasonalNaive` model. 

#### Produce forecasts

To generate forecasts we only have to use the `predict` method specifying the forecast horizon (`h`). In addition, to calculate prediction intervals associated to the forecasts, we can include the parameter `level` that receives a list of levels of the prediction intervals we want to build. In this case we will only calculate the 90% forecast interval (`level=[90]`). 

In [ ]:
forecasts = sf.predict(h=24, level=[90])
forecasts.head()

Let's look at our forecasts graphically.

In [ ]:
sf.plot(df, forecasts, max_insample_length=24 * 7)

### Performance of the MSTL model

#### Split Train/Test sets

To validate the accuracy of the `MSTL` model, we will show its performance on unseen data. We will use a classical time series technique that consists of dividing the data into a training set and a test set. We will leave the last 24 observations (the last day) as the test set. So the model will train on `32,872` observations. 

In [ ]:
df_test = df.tail(24)
df_train = df.drop(df_test.index)

#### MSTL model

In addition to the `MSTL` model, we will include the `SeasonalNaive` model as a benchmark to validate the added value of the `MSTL` model. Including `StatsForecast` models is as simple as adding them to the list of models to be fitted.

In [ ]:
sf = StatsForecast(
    models=[mstl, SeasonalNaive(season_length=24)], # add SeasonalNaive model to the list
    freq='H'
)

To measure the fitting time we will use the `time` module. 

In [ ]:
from time import time

To retrieve the forecasts of the test set we only have to do fit and predict as before.

In [ ]:
init = time()
sf = sf.fit(df=df_train)
forecasts_test = sf.predict(h=len(df_test), level=[90])
end = time()
forecasts_test.head()

In [ ]:
time_mstl = (end - init) / 60
print(f'MSTL Time: {time_mstl:.2f} minutes')

Then we were able to generate forecasts for the next 24 hours. Now let's look at the graphical comparison of the forecasts with the actual values.

Let's add the real value of test to our `forecasts_test` dataframe.

In [ ]:
forecasts_test = forecasts_test.merge(df_test, how='left', on=['unique_id', 'ds'])

In [ ]:
sf.plot(df_train, 
        forecasts_test, 
        models=['MSTL', 'SeasonalNaive'], 
        max_insample_length=24 * 7)

Let's look at those produced only by `MSTL`. 

In [ ]:
sf.plot(df_train, 
        forecasts_test, 
        models=['MSTL'], 
        max_insample_length=24 * 7)

We note that `MSTL` produces very accurate forecasts that follow the behavior of the time series. Now let us calculate numerically the accuracy of the model. We will use the following metrics: `MAE`, `MAPE`, `MASE`, `RMSE`, `SMAPE`.

In [ ]:
from datasetsforecast.losses import (
    mae, mape, mase, rmse, smape
)

In [ ]:
def evaluate_performace(y_hist, y_true, y_pred, models):
    if 'y' not in y_pred:
        y_true = y_true.merge(y_pred, how='left', on=['unique_id', 'ds'])
    else:
        y_true = y_pred
    evaluation = {}
    for model in models:
        evaluation[model] = {}
        for metric in [mase, mae, mape, rmse, smape]:
            metric_name = metric.__name__
            if metric_name == 'mase':
                evaluation[model][metric_name] = metric(y_true['y'].values, 
                                                 y_true[model].values, 
                                                 y_hist['y'].values, seasonality=24)
            else:
                evaluation[model][metric_name] = metric(y_true['y'].values, y_true[model].values)
    return pd.DataFrame(evaluation).T

In [ ]:
evaluate_performace(df_train, df_test, forecasts_test, models=['MSTL', 'SeasonalNaive'])

We observe that `MSTL` has an improvement of about 60% over the `SeasonalNaive` method in the test set measured in `MASE`.

#### Comparison with Prophet

One of the most widely used models for time series forecasting is `Prophet`. This model is known for its ability to model different seasonalities (weekly, daily yearly). We will use this model as a benchmark to see if the `MSTL` adds value for this time series. 

In [ ]:
from prophet import Prophet

# create prophet model
prophet = Prophet(interval_width=0.9)
init = time()
prophet.fit(df_train)
# produce forecasts
future = prophet.make_future_dataframe(periods=len(df_test), freq='H', include_history=False)
forecast_prophet = prophet.predict(future)
end = time()
# data wrangling
forecast_prophet = forecast_prophet[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
forecast_prophet.columns = ['ds', 'Prophet', 'Prophet-lo-90', 'Prophet-hi-90']
forecast_prophet.insert(0, 'unique_id', 'PJM_Load_hourly')
forecast_prophet.head()

In [ ]:
time_prophet = (end - init) / 60
print(f'Prophet Time: {time_prophet:.2f} minutes')

In [ ]:
times = pd.DataFrame({'model': ['MSTL', 'Prophet'], 'time (mins)': [time_mstl, time_prophet]})
times

We observe that the time required for `Prophet` to perform the fit and predict pipeline is greater than `MSTL`. Let's look at the forecasts produced by `Prophet`.

In [ ]:
forecasts_test = forecasts_test.merge(forecast_prophet, how='left', on=['unique_id', 'ds'])

In [ ]:
sf.plot(df_train, 
        forecasts_test, 
        models=['MSTL', 'SeasonalNaive', 'Prophet'],
        max_insample_length=24 * 7)

We note that `Prophet` is able to capture the overall behavior of the time series. However, in some cases it produces forecasts well below the actual value. It also does not correctly adjust the valleys. 

In [ ]:
evaluate_performace(df_train, df_test, forecasts_test, models=['MSTL', 'Prophet', 'SeasonalNaive'])

In terms of accuracy, `Prophet` is not able to produce better forecasts than the `SeasonalNaive` model, however, the `MSTL` model improves `Prophet`'s forecasts by 69% (`MASE`).

#### Comparison with NeuralProphet

`NeuralProphet` is the version of `Prophet` using deep learning. This model is also capable of handling different seasonalities so we will also use it as a benchmark. 

In [ ]:
from neuralprophet import NeuralProphet

neuralprophet = NeuralProphet(quantiles=[0.05, 0.95])
init = time()
neuralprophet.fit(df_train.drop(columns='unique_id'))
future = neuralprophet.make_future_dataframe(df=df_train.drop(columns='unique_id'), periods=len(df_test))
forecast_np = neuralprophet.predict(future)
end = time()
forecast_np = forecast_np[['ds', 'yhat1', 'yhat1 5.0%', 'yhat1 95.0%']]
forecast_np.columns = ['ds', 'NeuralProphet', 'NeuralProphet-lo-90', 'NeuralProphet-hi-90']
forecast_np.insert(0, 'unique_id', 'PJM_Load_hourly')
forecast_np.head()

In [ ]:
time_np = (end - init) / 60
print(f'Prophet Time: {time_np:.2f} minutes')

In [ ]:
times = times.append({'model': 'NeuralProphet', 'time (mins)': time_np}, ignore_index=True)
times

We observe that `NeuralProphet` requires a longer processing time than `Prophet` and `MSTL`.

In [ ]:
forecasts_test = forecasts_test.merge(forecast_np, how='left', on=['unique_id', 'ds'])

In [ ]:
sf.plot(df_train, 
        forecasts_test, 
        models=['MSTL', 'NeuralProphet', 'Prophet'],
        max_insample_length=24 * 7)

The forecasts graph shows that `NeuralProphet` generates very similar results to `Prophet`, as expected. 

In [ ]:
evaluate_performace(df_train, df_test, forecasts_test, models=['MSTL', 'NeuralProphet', 'Prophet', 'SeasonalNaive'])

With respect to numerical evaluation, `NeuralProphet` improves the results of `Prohet`, as expected, however, `MSTL` improves over `NeuralProphet`'s foreacasts by 68% (`MASE`).

:::{.callout-important}
The performance of `NeuralProphet` can be improved using hyperparameter optimization, which can increase the fitting time significantly. In this example we show its performance with the default version.
:::

## Conclusion

In this post we introduced `MSTL`, a model originally developed by [Kasun Bandara, Rob Hyndman and Christoph Bergmeir](https://arxiv.org/abs/2107.13462) capable of handling time series with multiple seasonalities. We also showed that for the PJM electricity load time series offers better performance in time and accuracy than the `Prophet` and `NeuralProphet` models.

## References

- [Bandara, Kasun & Hyndman, Rob & Bergmeir, Christoph. (2021). “MSTL: A Seasonal-Trend Decomposition Algorithm for Time Series with Multiple Seasonal Patterns”](https://arxiv.org/abs/2107.13462).

